# Gun 3 - Model Gecislerinin Loglanarak Test Edilmesi (DOC-28)

Amac (ticket'tan): *"Gecislerin sorunsuz calistigi loglanarak test edilecek."* ve **uctan uca zincir #3**:
OCR -> RAG -> Siniflandirma zincirini hem cloud hem local model konfigurasyonuyla bir kez calistirip, 31 Agustos'taki asil birlestirme gununden (DOC-29) once multi-model kirilganligini erken yakalamak.

Bu notebook'ta:

1. `src/llm_factory.py`'e DOC-28 kapsaminda eklenen loglamayi (her `generate()` cagrisi icin saglayici/model/sure/basari-hata) canli olarak yakalayip yapisal bir rapora ceviriyoruz.
2. Gercek OCR ciktilariyla (`data/processed/ocr_real_outputs.json`) tam zinciri (split -> embed -> classify -> index -> search) **once cloud (Anthropic) konfigurasyonuyla, sonra local (huggingface) konfigurasyonuyla** calistiriyoruz.
3. Local tarafta, Gun 2'de (notebook 11) kullanilan cok kucuk (135M) modelden daha yetkin ama hala CPU'da makul hizda calisan `Qwen/Qwen2.5-0.5B-Instruct` kullaniyoruz (config'teki gated/8B varsayilan yerine).
4. Sonuclari `data/processed/multi_model_chain_report.json`'a kaydedip, iki konfigurasyonu (basari orani, ortalama sure, hatalar) karsilastiriyoruz.

In [1]:
import sys
import os
import json
import time
import logging

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

from text_splitter import split_text
from embedder import embed_chunks
from vector_store import build_index, search
from classifier import classify_document, attach_labels_to_chunks
from llm_factory import get_llm_client, load_llm_config

print("Moduller yuklendi.")

Moduller yuklendi.


## 1. Loglamayi yakalama duzenegi

In [2]:
class ListLogHandler(logging.Handler):
    """llm_factory logger'indaki (DOC-28) generate() basladi/tamamlandi/basarisiz
    kayitlarini bellekte bir listeye toplar, boylece rapor JSON'ina yazilabilir."""

    def __init__(self):
        super().__init__()
        self.records: list[dict] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.records.append({
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        })


llm_logger = logging.getLogger("llm_factory")
llm_logger.setLevel(logging.INFO)
llm_logger.propagate = False  # root logger'a tekrar dusmesin (cift basim olmasin)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s %(name)s %(levelname)s %(message)s"))
list_handler = ListLogHandler()

llm_logger.handlers = [console_handler, list_handler]

print("llm_factory logger'i INFO seviyesinde, konsol + bellek-listesi handler'lariyla kuruldu.")

llm_factory logger'i INFO seviyesinde, konsol + bellek-listesi handler'lariyla kuruldu.


## 2. Gercek OCR verisini yukle

In [3]:
OCR_PATH = "../data/processed/ocr_real_outputs.json"
with open(OCR_PATH, encoding="utf-8") as f:
    ocr_outputs = json.load(f)


def format_document(fields: dict) -> str:
    return (
        f"Talep Eden: {fields['talep_eden']}\n"
        f"Tarih: {fields['tarih']}\n"
        f"Departman: {fields['departman']}\n"
        f"Konu: {fields['konu']}\n\n"
        f"{fields['aciklama']}"
    )


print(f"{len(ocr_outputs)} belge yuklendi: {list(ocr_outputs.keys())}")

5 belge yuklendi: ['test_talep_01.png', 'test_talep_02.png', 'test_talep_03.png', 'test_talep_04.png', 'test_talep_05.png']


## 3. Uctan uca zincir fonksiyonu (OCR -> RAG -> Siniflandirma)

Her konfigurasyon (cloud/local) icin: her belgeyi `split_text` + `embed_chunks` ile RAG'a hazirla, `classify_document`'i o konfigurasyonun `LLMClient`'iyla (Factory'den) calistir, hatalari yutmadan kaydet, en sonda in-memory bir FAISS index kurup bir ornek arama ile RAG'in da sorunsuz calistigini dogrula.

RAG tarafi (`embedder.py`, her zaman yerel `sentence-transformers`) cloud/local LLM secimine bagli DEGIL; sadece siniflandirma adimi Factory uzerinden degisiyor. Bu yuzden bu test zincirin tamaminin (OCR->RAG->Siniflandirma) her iki LLM konfigurasyonuyla da sorunsuz akip akmadigini gosteriyor.

In [4]:
def run_chain(config_label: str, llm_settings: dict) -> dict:
    print(f"\n{'='*70}\n{config_label.upper()} konfigurasyonuyla zincir basliyor\n{'='*70}")

    list_handler.records.clear()
    client = get_llm_client(llm_settings)

    classifications = {}
    classification_errors = []
    all_chunks = []
    chain_t0 = time.time()

    for filename, fields in sorted(ocr_outputs.items()):
        document_text = format_document(fields)

        chunks = split_text(document_text)
        for c in chunks:
            c["source_doc"] = filename
        embedded = embed_chunks(chunks)
        all_chunks.extend(embedded)

        try:
            result = classify_document(document_text, client=client)
            classifications[filename] = result
            print(f"  OK  {filename}: siniflar={result['siniflar']} guven={result.get('guven')}")
        except Exception as e:
            classification_errors.append({"filename": filename, "error": f"{type(e).__name__}: {e}"})
            print(f"  HATA {filename}: {type(e).__name__}: {e}")

    chain_duration = time.time() - chain_t0

    labeled_chunks = attach_labels_to_chunks(all_chunks, classifications)
    index, metadata = build_index(labeled_chunks)
    query = "monitor talebi"
    query_embedding = embed_chunks([{"chunk_id": 0, "text": query, "token_count": 0}])[0]["embedding"]
    search_results = search(index, metadata, query_embedding, top_k=1)
    search_ok = len(search_results) > 0

    return {
        "config_label": config_label,
        "active_mode": llm_settings["active_mode"],
        "provider": type(client).__name__,
        "model_name": client.model_name,
        "documents_total": len(ocr_outputs),
        "classification_success": len(classifications),
        "classification_errors": classification_errors,
        "chain_duration_sec": round(chain_duration, 2),
        "rag_chunks_indexed": len(labeled_chunks),
        "search_smoke_test": {
            "query": query,
            "ok": search_ok,
            "top_result": search_results[0] if search_ok else None,
        },
        "log_events": list(list_handler.records),
    }

## 4. Zinciri cloud konfigurasyonuyla calistir

In [5]:
cloud_settings = load_llm_config()  # settings.yaml -> active_mode: cloud (anthropic)
report_cloud = run_chain("cloud", cloud_settings)
print(f"\nCloud ozet: {report_cloud['classification_success']}/{report_cloud['documents_total']} basarili, "
      f"sure={report_cloud['chain_duration_sec']}sn, arama_ok={report_cloud['search_smoke_test']['ok']}")

2026-08-25 13:09:43,950 llm_factory INFO get_llm_client: active_mode=cloud provider=anthropic model_name=claude-sonnet-5 -> AnthropicClient



CLOUD konfigurasyonuyla zincir basliyor


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-25 13:09:51,179 llm_factory INFO generate basladi: provider=AnthropicClient model=claude-sonnet-5 max_tokens=512


2026-08-25 13:09:54,062 llm_factory INFO generate tamamlandi: provider=AnthropicClient model=claude-sonnet-5 sure=2.88sn yanit_uzunlugu=225


...

2026-08-25 13:10:02,153 llm_factory INFO generate tamamlandi: provider=AnthropicClient model=claude-sonnet-5 sure=2.54sn yanit_uzunlugu=248


2026-08-25 13:10:02,199 llm_factory INFO generate basladi: provider=AnthropicClient model=claude-sonnet-5 max_tokens=512


  OK  test_talep_04.png: siniflar=['talep formu'] guven=0.9


2026-08-25 13:10:05,129 llm_factory INFO generate tamamlandi: provider=AnthropicClient model=claude-sonnet-5 sure=2.93sn yanit_uzunlugu=237


  OK  test_talep_05.png: siniflar=['talep formu'] guven=0.9

Cloud ozet: 5/5 basarili, sure=20.07sn, arama_ok=True


## 5. Zinciri local konfigurasyonuyla calistir

Config'teki varsayilan local model (`meta-llama/Meta-Llama-3-8B-Instruct`) gated ve bu ortamda (CPU-only, HF_TOKEN yok) erisilemiyor (bkz. notebook 10, bolum 4). Bu yuzden burada, notebook 11'de kullanilan cok kucuk (135M) modelden daha yetkin ama hala CPU'da makul hizda calisan **`Qwen/Qwen2.5-0.5B-Instruct`** (herkese acik) kullaniliyor. Kod `local_model.model_name` disinda hicbir yerde degismiyor -- ayni `Factory + classify_document` entegrasyonu.

In [6]:
local_settings = {
    "active_mode": "local",
    "local_model": {"provider": "huggingface", "model_name": "Qwen/Qwen2.5-0.5B-Instruct"},
}
report_local = run_chain("local", local_settings)
print(f"\nLocal ozet: {report_local['classification_success']}/{report_local['documents_total']} basarili, "
      f"sure={report_local['chain_duration_sec']}sn, arama_ok={report_local['search_smoke_test']['ok']}")

2026-08-25 13:10:05,163 llm_factory INFO get_llm_client: active_mode=local provider=huggingface model_name=Qwen/Qwen2.5-0.5B-Instruct -> LocalHFClient


2026-08-25 13:10:05,205 llm_factory INFO generate basladi: provider=LocalHFClient model=Qwen/Qwen2.5-0.5B-Instruct max_tokens=512



LOCAL konfigurasyonuyla zincir basliyor


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

2026-08-25 13:10:19,756 llm_factory INFO generate tamamlandi: provider=LocalHFClient model=Qwen/Qwen2.5-0.5B-Instruct sure=14.55sn yanit_uzunlugu=151


...

2026-08-25 13:10:56,399 llm_factory INFO generate tamamlandi: provider=LocalHFClient model=Qwen/Qwen2.5-0.5B-Instruct sure=11.72sn yanit_uzunlugu=143


2026-08-25 13:10:56,418 llm_factory INFO generate basladi: provider=LocalHFClient model=Qwen/Qwen2.5-0.5B-Instruct max_tokens=512


  OK  test_talep_04.png: siniflar=['diğer'] guven=0.9


2026-08-25 13:11:08,340 llm_factory INFO generate tamamlandi: provider=LocalHFClient model=Qwen/Qwen2.5-0.5B-Instruct sure=11.92sn yanit_uzunlugu=156


  OK  test_talep_05.png: siniflar=['dilekçe', 'talep formu'] guven=0.7

Local ozet: 5/5 basarili, sure=63.18sn, arama_ok=True


## 6. Raporu kaydet ve karsilastir

In [7]:
REPORT_PATH = "../data/processed/multi_model_chain_report.json"
combined_report = {"cloud": report_cloud, "local": report_local}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(combined_report, f, ensure_ascii=False, indent=2)

print(f"Rapor kaydedildi -> {REPORT_PATH}\n")

print(f"{'Konfigurasyon':<10} {'Provider':<16} {'Model':<32} {'Basari':<10} {'Sure(sn)':<10} {'RAG arama'}")
for label, r in combined_report.items():
    basari = f"{r['classification_success']}/{r['documents_total']}"
    print(f"{label:<10} {r['provider']:<16} {r['model_name']:<32} {basari:<10} {r['chain_duration_sec']:<10} {r['search_smoke_test']['ok']}")

for label, r in combined_report.items():
    assert r["rag_chunks_indexed"] > 0, f"{label}: RAG zinciri hic chunk indekslemedi"
    assert r["search_smoke_test"]["ok"], f"{label}: RAG arama smoke-testi basarisiz"

print("\nOK - RAG zinciri (split -> embed -> index -> search) HER IKI konfigurasyonda da sorunsuz calisti.")
print("Siniflandirma basari oranlari yukaridaki tabloda; asagida bulgular ozetleniyor.")

Rapor kaydedildi -> ../data/processed/multi_model_chain_report.json

Konfigurasyon Provider         Model                            Basari     Sure(sn)   RAG arama
cloud      AnthropicClient  claude-sonnet-5                  5/5        20.07      True
local      LocalHFClient    Qwen/Qwen2.5-0.5B-Instruct       5/5        63.18      True

OK - RAG zinciri (split -> embed -> index -> search) HER IKI konfigurasyonda da sorunsuz calisti.
Siniflandirma basari oranlari yukaridaki tabloda; asagida bulgular ozetleniyor.


## Bulgular / DOC-29 icin notlar

- **RAG zinciri** (OCR ciktisi -> split_text -> embed_chunks -> FAISS index -> arama), LLM saglayicisindan tamamen bagimsiz oldugu icin cloud/local her iki konfigurasyonda da sorunsuz calisti (yukaridaki assert'ler bunu dogruluyor).
- **Cloud (Anthropic) siniflandirma**: 5/5 belgede basarili JSON + dogru sinif (`talep formu`, `guven` 0.85-0.95) — notebook 08/09'daki sonuclarla tutarli, regresyon yok.
- **Local siniflandirma (`Qwen2.5-0.5B-Instruct`)**: 5/5 belgede **gecerli JSON** uretti (Gun 2'deki notebook 11'de 135M'lik cok daha kucuk modelin basaramadigi sey) — Factory/entegrasyon kodu acisindan "sorunsuz gecis" saglandi. Ancak **siniflandirma icerigi cloud'dan belirgin sekilde sapti**: 4/5 belge `talep formu` yerine `diğer` (fallback) olarak etiketlendi, bir belge de tek yerine iki sinifa (`dilekçe`, `talep formu`) atandi. Yani kucuk local model *format* olarak calisiyor ama *siniflandirma kalitesi* cloud modelinden gozle gorulur sekilde dusuk.
- **Sonuc**: DOC-29'daki asil birlestirmeden once dikkat edilmesi gereken kirilganlik, baglanti/format hatasi degil, **kucuk local modellerin siniflandirma dogrulugu** — uretimde local mod kullanilacaksa (config'teki gibi Llama-3-8B-Instruct boyutunda, instruction-tuned bir model onerilir; 0.5B'lik bir model dogruluk acisindan yeterli degil.
- **Loglama** (DOC-28): `llm_factory` logger'i her `generate()` cagrisi icin saglayici/model/sure/basari-hata bilgisini standart `logging` modulu ile uretiyor; bu notebook'ta hem konsola basildi hem de yapisal olarak `data/processed/multi_model_chain_report.json` -> `log_events` altina kaydedildi. Uretimde bu logger'a bir dosya/monitoring handler'i baglanarak model gecisleri izlenebilir.
- Config'teki varsayilan local model (gated `Meta-Llama-3-8B-Instruct`) bu ortamda test edilemedi (HF erisimi yok); `HF_TOKEN` eklenip erisim onaylandiginda ayni kod (sadece `model_name` gercek degeriyle) calisir — gercek hedef modelle dogruluk olcumu yapilamadigi icin bu da DOC-29 oncesi acik bir risk olarak not edilmeli.